In [1]:
import numpy as np
np.bool = np.bool_ 
import pandas as pd
import os
from tqdm import tqdm

import mxnet as mx
from mxnet import gluon
from mxnet import autograd
from mxnet import image

import sys

from sklearn.metrics import matthews_corrcoef

import random

import rasterio
from rasterio.enums import ColorInterp
from rasterio.features import shapes
from scipy import ndimage

import matplotlib.pyplot as plt
%matplotlib inline

import higra as hg
import torch
import pickle

import itertools

import imageio.v3 as imageio
from skimage.color import label2rgb
import cv2
from skimage.segmentation import mark_boundaries
from skimage.segmentation import find_boundaries

from shapely.geometry import shape
import shapely
from shapely import Polygon, MultiPolygon, orient_polygons
from shapely.wkt import loads
from shapely import ops
from shapely.ops import polygonize

import geopandas as gpd
import pyproj

import ast

import glob

import multiprocessing

In [2]:
data = pd.read_csv('inference_data/inference_data.csv')
image_filenames = data[data["num_cultivated_pixels"] > 0]["image_filename"].tolist() # images that have at least one cultivated pixel

# Combine all field predictions into 2 files 

In [3]:
filenames = [f'inference_data/outputs/field_predictions/{image_filename[-9:-4]}.csv' for image_filename in image_filenames]

In [4]:
# https://stackoverflow.com/questions/75756119/concatenate-10-000-csv-files-in-a-directory-using-python-pandas-too-slow
with open("inference_data/outputs/FieldSetsUS_1.csv", "w") as output:
  first = True
  for filename in tqdm(filenames[:26000], smoothing=0):
    with open(filename, "r") as inputfile:
      for no, line in enumerate(inputfile, 1):
        if no == 1 and not first:
          continue
        first = False
        output.write(line)

100%|██████████| 26000/26000 [1:35:38<00:00,  4.53it/s]


In [ ]:
# https://stackoverflow.com/questions/75756119/concatenate-10-000-csv-files-in-a-directory-using-python-pandas-too-slow
with open("inference_data/outputs/FieldSetsUS_2.csv", "w") as output:
  first = True
  for filename in tqdm(filenames[26000:], smoothing=0):
    with open(filename, "r") as inputfile:
      for no, line in enumerate(inputfile, 1):
        if no == 1 and not first:
          continue
        first = False
        output.write(line)

# Combine Single Best Parameter predictions into one file

In [3]:
header = True
for df in tqdm(pd.read_csv("inference_data/outputs/FieldSetsUS_1.csv", chunksize=1000000)):
    filtered_df = df[df['priority_ranking']==1] # T=0.08 is the single best parameter
    filtered_df.to_csv('inference_data/outputs/Single_Best_Parameter_map.csv', mode='a', index=False, header=header)
    header = False

for df in tqdm(pd.read_csv("inference_data/outputs/FieldSetsUS_2.csv", chunksize=1000000)):
    filtered_df = df[df['priority_ranking']==1] # T=0.08 is the single best parameter
    filtered_df.to_csv('inference_data/outputs/Single_Best_Parameter_map.csv', mode='a', index=False, header=header)

34it [24:15, 42.81s/it]
31it [23:34, 45.64s/it]
